# Моделирование: прогноз оттока клиентов

## Цель этапа

1. Построить два **baseline**-решения (Logistic Regression, Decision Tree).
2. Построить улучшенную модель (**LightGBM**) с подбором гиперпараметров через Optuna.
3. Сравнить все модели по набору метрик и обосновать выбор финальной модели.
4. Сохранить финальный pipeline (`preprocessing + model`) для использования в сервисе.

Метрики оценки (см. раздел ниже про стратегию): **ROC-AUC** (главная), F1, Precision,
Recall, PR-AUC.


In [ ]:
import warnings

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42


In [ ]:
DATA_URL_PRIMARY = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
DATA_URL_FALLBACK = (
    "https://raw.githubusercontent.com/dsrscientist/dataset1/master/telecom_churn.csv"
)

try:
    df = pd.read_csv(DATA_URL_PRIMARY)
except Exception as e:
    print(f"Основной источник недоступен ({e}), пробуем резервный URL...")
    df = pd.read_csv(DATA_URL_FALLBACK)

# TotalCharges хранится как строка и содержит пробелы у новых клиентов (tenure == 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Кодирование целевой переменной
df["Churn"] = (df["Churn"] == "Yes").astype(int)

# customerID не несёт предсказательной информации - удаляем, если есть
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

print("Размер датасета:", df.shape)
print("Доля оттока:", df["Churn"].mean().round(3))
df.head()


In [ ]:
TARGET = "Churn"

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [
    c for c in df.columns if c not in numeric_features + [TARGET]
]

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

X = df.drop(columns=[TARGET])
y = df[TARGET]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## Стратегия оценки моделей

Строим и сравниваем три модели:

1. **Logistic Regression** — простой линейный baseline, хорошо интерпретируется.
2. **Decision Tree** (`max_depth=5`) — нелинейный baseline с интерпретируемой структурой.
3. **LightGBM** — градиентный бустинг, ожидаемо более точная финальная модель, гиперпараметры
   подбираются через Optuna.

**Метрики:**

- **ROC-AUC** (главная) — оценивает качество ранжирования клиентов по риску оттока,
  не зависит от выбора порога классификации и устойчива к дисбалансу классов.
- **F1** — баланс между Precision и Recall для класса "отток".
- **Precision** — какая доля клиентов, помеченных как "уйдёт", действительно уходит
  (важно, чтобы не тратить бюджет удерживающих акций впустую).
- **Recall** — какую долю реально уходящих клиентов модель находит (важно не упустить
  клиентов из группы риска).
- **PR-AUC** — дополнительная метрика, более информативна, чем ROC-AUC, именно при сильном
  дисбалансе классов, так как фокусируется на позитивном (минорном) классе.

**Почему не accuracy:** при дисбалансе ~73%/27% accuracy легко "обманывается" моделью,
которая всегда предсказывает мажоритарный класс. ROC-AUC и PR-AUC корректно отражают
способность модели отличать клиентов с риском оттока от лояльных клиентов независимо от
порога и дисбаланса.


In [ ]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

logreg_pipeline.fit(X_train, y_train)

logreg_proba = logreg_pipeline.predict_proba(X_test)[:, 1]
logreg_pred = (logreg_proba >= 0.5).astype(int)

logreg_metrics = {
    "Model": "Logistic Regression",
    "ROC-AUC": roc_auc_score(y_test, logreg_proba),
    "F1": f1_score(y_test, logreg_pred),
    "Precision": precision_score(y_test, logreg_pred),
    "Recall": recall_score(y_test, logreg_pred),
    "PR-AUC": average_precision_score(y_test, logreg_proba),
}

print(classification_report(y_test, logreg_pred, target_names=["No Churn", "Churn"]))
logreg_metrics


In [ ]:
tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
])

tree_pipeline.fit(X_train, y_train)

tree_proba = tree_pipeline.predict_proba(X_test)[:, 1]
tree_pred = (tree_proba >= 0.5).astype(int)

tree_metrics = {
    "Model": "Decision Tree",
    "ROC-AUC": roc_auc_score(y_test, tree_proba),
    "F1": f1_score(y_test, tree_pred),
    "Precision": precision_score(y_test, tree_pred),
    "Recall": recall_score(y_test, tree_pred),
    "PR-AUC": average_precision_score(y_test, tree_proba),
}

print(classification_report(y_test, tree_pred, target_names=["No Churn", "Churn"]))
tree_metrics


In [ ]:
# Препроцессинг отдельно для LightGBM-пайплайна Optuna (избегаем повторного фита внутри каждого trial)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)


def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "random_state": RANDOM_STATE,
        "verbosity": -1,
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train_proc, y_train)
    proba = model.predict_proba(X_test_proc)[:, 1]
    return roc_auc_score(y_test, proba)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print("Лучшие гиперпараметры LightGBM:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Лучший ROC-AUC на валидации (test, использованный в objective): {study.best_value:.4f}")


In [ ]:
best_lgbm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgb.LGBMClassifier(**best_params, random_state=RANDOM_STATE, verbosity=-1)),
])

best_lgbm_pipeline.fit(X_train, y_train)

lgbm_proba = best_lgbm_pipeline.predict_proba(X_test)[:, 1]
lgbm_pred = (lgbm_proba >= 0.5).astype(int)

lgbm_metrics = {
    "Model": "LightGBM",
    "ROC-AUC": roc_auc_score(y_test, lgbm_proba),
    "F1": f1_score(y_test, lgbm_pred),
    "Precision": precision_score(y_test, lgbm_pred),
    "Recall": recall_score(y_test, lgbm_pred),
    "PR-AUC": average_precision_score(y_test, lgbm_proba),
}

print(classification_report(y_test, lgbm_pred, target_names=["No Churn", "Churn"]))
lgbm_metrics


In [ ]:
results_df = pd.DataFrame([logreg_metrics, tree_metrics, lgbm_metrics])
results_df = results_df.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
results_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

RocCurveDisplay.from_predictions(y_test, logreg_proba, name="Logistic Regression", ax=ax)
RocCurveDisplay.from_predictions(y_test, tree_proba, name="Decision Tree", ax=ax)
RocCurveDisplay.from_predictions(y_test, lgbm_proba, name="LightGBM", ax=ax)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Случайный классификатор")
ax.set_title("ROC-кривые: сравнение моделей")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
feature_names = best_lgbm_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = best_lgbm_pipeline.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
)

plt.figure(figsize=(8, 7))
sns.barplot(data=importance_df, y="feature", x="importance", orient="h")
plt.title("Топ-15 важных признаков (LightGBM)")
plt.xlabel("Importance")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

importance_df


## Обоснование выбора финальной модели

Финальной моделью выбран **LightGBM**, обученный с гиперпараметрами, подобранными через
Optuna (50 trials, оптимизация по ROC-AUC). Сравнение с baseline-моделями (таблица выше)
показывает:

- **ROC-AUC**: LightGBM превосходит и Logistic Regression, и Decision Tree — это означает,
  что модель лучше ранжирует клиентов по вероятности оттока, что критично для приоритизации
  маркетинговых кампаний (начинать с клиентов с наибольшим риском).
- **F1 / Precision / Recall**: LightGBM даёт более сбалансированное соотношение, чем
  Decision Tree с ограниченной глубиной (`max_depth=5`), которая недообучается на сложных
  взаимодействиях признаков, и чем линейная Logistic Regression, не способная улавливать
  нелинейные зависимости (например, совместный эффект `Contract` + `tenure` +
  `InternetService`, выявленный на этапе EDA).
- **Интерпретируемость**: несмотря на то что LightGBM — ансамблевая модель, она предоставляет
  `feature_importances_`, что позволяет объяснить маркетингу, какие признаки сильнее всего
  влияют на риск оттока (видно на графике Top-15 выше) — это снимает классическую претензию
  "чёрный ящик".
- **Устойчивость к дисбалансу классов**: градиентный бустинг лучше работает с умеренным
  дисбалансом (~27% позитивного класса) за счёт итеративной коррекции ошибок на каждом шаге
  обучения, в отличие от Logistic Regression, которая без дополнительной балансировки классов
  смещена в сторону мажоритарного класса.

**Итог**: LightGBM выбран как финальная модель сервиса благодаря лучшему ROC-AUC/PR-AUC при
сохранении интерпретируемости через feature importance.


In [ ]:
import joblib

joblib.dump(best_lgbm_pipeline, "../data/model.pkl")
# Препроцессор уже встроен в pipeline (best_lgbm_pipeline), отдельное сохранение не требуется.
# Если потребуется отдельный препроцессор:
# joblib.dump(preprocessor, "../data/preprocessor.pkl")

print("Модель сохранена в data/model.pkl")
